In [1]:
import pandas as pd
import numpy as np
import requests

DATA_PATH = '../data'

full_dataset = pd.read_parquet(f'{DATA_PATH}/full_dataset.parquet')
movies = pd.read_csv(f'{DATA_PATH}/movies.csv')

print("Loaded successfully")
print(full_dataset.shape)

Loaded successfully
(50000190, 34)


In [2]:
import requests

# Get weather for Rochester, NY ( my location )
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 43.1566,
    "longitude": -77.6088,
    "current_weather": True
}
response = requests.get(url, params=params)
print(response.json())

{'latitude': 43.15501, 'longitude': -77.596855, 'generationtime_ms': 0.14710426330566406, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 161.0, 'current_weather_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature': '°C', 'windspeed': 'km/h', 'winddirection': '°', 'is_day': '', 'weathercode': 'wmo code'}, 'current_weather': {'time': '2026-03-30T01:45', 'interval': 900, 'temperature': 7.6, 'windspeed': 5.7, 'winddirection': 108, 'is_day': 0, 'weathercode': 3}}


In [3]:
def get_weather_condition(weathercode, temperature):
    if weathercode in [51, 53, 55, 61, 63, 65, 80, 81, 82, 95, 96, 99]:
        return 'rainy'
    elif weathercode in [71, 73, 75, 77, 85, 86]:
        return 'snowy'
    elif weathercode in [0, 1] and temperature >= 28:
        return 'sunny'
    else:
        return None  # normal day — no weather row

condition = get_weather_condition(3, 7.5)
print(f"Condition: {condition}")

Condition: None


In [4]:
# Map weather condition to boosted genres
weather_genre_map = {
    'rainy': ['Drama', 'Romance', 'Thriller', 'Mystery', 'Film-Noir'],
    'snowy': ['Animation', 'Children', 'Fantasy', 'Musical', 'Comedy'],
    'sunny': ['Action', 'Adventure', 'Comedy']
}

def get_weather_recommendations(condition, full_dataset, movies, n=10):
    
    # No special weather — return nothing
    if condition is None:
        return None
    
    # Get boosted genres for this weather
    boosted_genres = weather_genre_map[condition]
    
    # Filter movies that match at least one boosted genre
    # and have enough ratings (quality filter)
    movie_features = full_dataset.drop_duplicates('movieId')
    movie_features = movie_features[movie_features['movie_rating_count'] >= 500]
    
    # Check if movie has any of the boosted genres
    genre_mask = movie_features[boosted_genres].sum(axis=1) > 0
    weather_candidates = movie_features[genre_mask].copy()
    
    # Score = avg rating × genre match count (how many boosted genres it has)
    weather_candidates['genre_match'] = weather_candidates[boosted_genres].sum(axis=1)
    weather_candidates['weather_score'] = (
        weather_candidates['movie_avg_rating'] * 
        weather_candidates['genre_match']
    )
    
    # Get top N
    top_n = weather_candidates.nlargest(n, 'weather_score')[['movieId', 'weather_score']]
    top_n['movieId'] = top_n['movieId'].astype(str)
    movies['movieId'] = movies['movieId'].astype(str)
    top_n = top_n.merge(movies, on='movieId')
    
    return top_n[['title', 'genres', 'weather_score']]

# Test with rainy condition
weather_recs = get_weather_recommendations('rainy', full_dataset, movies)
print(weather_recs)

                                               title  \
0                                     Vertigo (1958)   
1                                     Rebecca (1940)   
2                                        Diva (1981)   
3  Secret in Their Eyes, The (El secreto de sus o...   
4  Three Days of the Condor (3 Days of the Condor...   
5                           In a Lonely Place (1950)   
6                                       Gilda (1946)   
7                            Mulholland Drive (2001)   
8                       Foreign Correspondent (1940)   
9                                Lost Highway (1997)   

                                          genres  weather_score  
0                 Drama|Mystery|Romance|Thriller      16.483286  
1                 Drama|Mystery|Romance|Thriller      16.280938  
2          Action|Drama|Mystery|Romance|Thriller      16.021488  
3           Crime|Drama|Mystery|Romance|Thriller      15.969271  
4                 Drama|Mystery|Romance|Thriller     

In [5]:
print("SNOWY DAY:")
print(get_weather_recommendations('snowy', full_dataset, movies))
print("\nHOT SUNNY DAY:")
print(get_weather_recommendations('sunny', full_dataset, movies))

SNOWY DAY:
                   title                                             genres  \
0         Tangled (2010)  Animation|Children|Comedy|Fantasy|Musical|Roma...   
1       Enchanted (2007)  Adventure|Animation|Children|Comedy|Fantasy|Mu...   
2          Presto (2008)                  Animation|Children|Comedy|Fantasy   
3      Inside Out (2015)  Adventure|Animation|Children|Comedy|Drama|Fantasy   
4   Partly Cloudy (2009)                  Animation|Children|Comedy|Fantasy   
5       Toy Story (1995)        Adventure|Animation|Children|Comedy|Fantasy   
6     Toy Story 3 (2010)   Adventure|Animation|Children|Comedy|Fantasy|IMAX   
7           Moana (2016)        Adventure|Animation|Children|Comedy|Fantasy   
8  Monsters, Inc. (2001)        Adventure|Animation|Children|Comedy|Fantasy   
9     Toy Story 2 (1999)        Adventure|Animation|Children|Comedy|Fantasy   

   weather_score  
0      18.666667  
1      17.403472  
2      15.791328  
3      15.729602  
4      15.700201  
5    

In [6]:
import pickle

DATA_PATH = '../data'

with open(f'{DATA_PATH}/xgb_model_v2.pkl', 'rb') as f:
    propensity_model = pickle.load(f)

with open(f'{DATA_PATH}/svd_model.pkl', 'rb') as f:
    svd_model = pickle.load(f)

print("Models loaded!")

Models loaded!


In [7]:
def time_relevance_score(user_peak_hour, user_peak_season,
                         movie_peak_hour, movie_peak_season,
                         user_rating_count):
    pattern_strength = min(user_rating_count / 200, 1.0)
    hour_match = 1.0 if abs(user_peak_hour - movie_peak_hour) <= 2 else 0.8
    season_match = 1.0 if user_peak_season == movie_peak_season else 0.9
    raw_score = hour_match * season_match
    nudge = 1.0 + (raw_score - 1.0) * pattern_strength
    return nudge

In [9]:
def get_weather_recommendations(user_id, condition, propensity_model, svd_model,
                                full_dataset, movies, n=10):
    if condition is None:
        return None

    boosted_genres = weather_genre_map[condition]

    # Get user features
    user_data = full_dataset[full_dataset['userId'] == user_id].iloc[0]
    user_peak_hour = user_data['user_peak_hour']
    user_peak_season = user_data['user_peak_season']
    user_rating_count = user_data['user_rating_count']

    # Already watched
    watched_movies = full_dataset[
        (full_dataset['userId'] == user_id) &
        (full_dataset['watched'] == 1)
    ]['movieId'].tolist()

    # Candidates — not watched + popular
    candidates = full_dataset[
        ~full_dataset['movieId'].isin(watched_movies)
    ].drop_duplicates('movieId').copy()
    candidates = candidates[candidates['movie_rating_count'] >= 500]

    # Signal 1 — Propensity
    drop_cols = ['userId', 'movieId', 'watched', 'user_peak_season',
                 'movie_peak_season', 'movie_rating_count', 'user_rating_count']
    X_candidates = candidates.drop(columns=drop_cols).fillna(0)
    candidates['propensity_score'] = propensity_model.predict_proba(X_candidates)[:, 1]

    # Signal 2 — SVD
    user_idx = svd_model.train_set.uid_map.get(str(user_id))
    if user_idx is not None:
        svd_scores = svd_model.score(user_idx)
        idx_to_item = {v: k for k, v in svd_model.train_set.iid_map.items()}
        candidates['svd_score'] = candidates['movieId'].astype(str).map(
            {idx_to_item[i]: svd_scores[i] for i in range(len(svd_scores))}
        ).fillna(svd_scores.mean())
        candidates['svd_score'] = (candidates['svd_score'] - svd_scores.min()) / (svd_scores.max() - svd_scores.min())
    else:
        candidates['svd_score'] = 0.5

    # Signal 3 — Time relevance
    candidates['time_score'] = candidates.apply(
        lambda row: time_relevance_score(
            user_peak_hour, user_peak_season,
            row['movie_peak_hour'], row['movie_peak_season'],
            user_rating_count
        ), axis=1
    )

    # Final score from ranking layer
    candidates['final_score'] = (
        candidates['propensity_score'] *
        candidates['svd_score'] *
        candidates['time_score']
    )

    # Signal 4 — Weather boost 30%
    candidates['weather_boost'] = candidates[boosted_genres].sum(axis=1).apply(
        lambda x: 1.3 if x > 0 else 1.0
    )

    # Weather score = final score × weather boost
    candidates['weather_score'] = candidates['final_score'] * candidates['weather_boost']

    # Top N
    top_n = candidates.nlargest(n, 'weather_score')[['movieId', 'propensity_score',
                                                      'svd_score', 'time_score',
                                                      'weather_boost', 'weather_score']]
    top_n['movieId'] = top_n['movieId'].astype(str)
    movies['movieId'] = movies['movieId'].astype(str)
    top_n = top_n.merge(movies, on='movieId')

    return top_n[['title', 'genres', 'propensity_score', 'svd_score',
                  'time_score', 'weather_boost', 'weather_score']]

# Test with rainy condition for user 1
results = get_weather_recommendations(1, 'rainy', propensity_model, svd_model,
                                       full_dataset, movies)
print(results)

                                               title  \
0                   Shawshank Redemption, The (1994)   
1        Seven Samurai (Shichinin no samurai) (1954)   
2                              Mulholland Dr. (1999)   
3                         Usual Suspects, The (1995)   
4  Lives of Others, The (Das leben der Anderen) (...   
5  Scenes From a Marriage (Scener ur ett äktenska...   
6                                Donnie Darko (2001)   
7                                  Fight Club (1999)   
8                              Third Man, The (1949)   
9                        Over the Garden Wall (2013)   

                          genres  propensity_score  svd_score  time_score  \
0                    Crime|Drama          0.965558   0.950867       0.902   
1         Action|Adventure|Drama          0.961778   0.950665       0.902   
2          Drama|Mystery|Romance          0.945735   0.962259       0.902   
3         Crime|Mystery|Thriller          0.977404   0.930230       0.902  

In [10]:
# Test with different users
print("User 500:")
print(get_weather_recommendations(500, 'rainy', propensity_model, svd_model, full_dataset, movies)[['title', 'genres']])

print("\nUser 1000:")
print(get_weather_recommendations(1000, 'rainy', propensity_model, svd_model, full_dataset, movies)[['title', 'genres']])

User 500:
                                               title  \
0                   Shawshank Redemption, The (1994)   
1                         Princess Bride, The (1987)   
2  Howl's Moving Castle (Hauru no ugoku shiro) (2...   
3                              Third Man, The (1949)   
4                City of God (Cidade de Deus) (2002)   
5  Scenes From a Marriage (Scener ur ett äktenska...   
6        Seven Samurai (Shichinin no samurai) (1954)   
7                        Over the Garden Wall (2013)   
8         Life Is Beautiful (La Vita è bella) (1997)   
9                              Mulholland Dr. (1999)   

                                    genres  
0                              Crime|Drama  
1  Action|Adventure|Comedy|Fantasy|Romance  
2      Adventure|Animation|Fantasy|Romance  
3               Film-Noir|Mystery|Thriller  
4    Action|Adventure|Crime|Drama|Thriller  
5                                    Drama  
6                   Action|Adventure|Drama  
7            